In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import arviz as az
import cmdstanpy as cs
import json, time, shutil

# Reproducibility
SEED        = 20251108
CHAINS      = 4
WARMUP      = 1000
DRAWS       = 1000
THIN        = 1
PARALLEL    = min(CHAINS, 4)

# Paths
DATA_PATH = Path("./_preprocessed/pa_long_trials.csv")
STAN_DIR = Path("./stan");     STAN_DIR.mkdir(parents=True, exist_ok=True)
RES_DIR  = Path("./results");  RES_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import cmdstanpy as cs
cs.rebuild_cmdstan()   # rebuilds the precompiled headers (PCH)

In [ ]:
# ---------- Data loader (keep original labels) ----------
def load_long(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    if "Unnamed: 0" in df.columns:
        df = df.drop(columns=["Unnamed: 0"])

    need = {"ID","subject_id","session","modality","group","trial","y"}
    missing = need - set(df.columns)
    assert not missing, f"Missing columns: {missing}"

    # subject index
    if df["subject_id"].min() == 1 and df["subject_id"].dtype.kind in "iu":
        df["subject_id"] = df["subject_id"].astype(int)
    else:
        df["subject_id"] = pd.Categorical(df["ID"]).codes + 1

    # keep original categorical labels & ordering
    sess_levels = list(pd.unique(df["session"]))
    mod_levels  = list(pd.unique(df["modality"]))
    grp_levels  = list(pd.unique(df["group"]))

    df["session"]  = pd.Categorical(df["session"],  categories=sess_levels, ordered=True)
    df["modality"] = pd.Categorical(df["modality"], categories=mod_levels,  ordered=True)
    df["group"]    = pd.Categorical(df["group"],    categories=grp_levels,  ordered=True)
    return df

# ---------- Token & design builders (same logic as NP) ----------
from itertools import product

def factor_tokens(df: pd.DataFrame):
    s_lvls = list(df["session"].cat.categories)
    m_lvls = list(df["modality"].cat.categories)
    g_lvls = list(df["group"].cat.categories)
    s_base, m_base, g_base = s_lvls[0], m_lvls[0], g_lvls[0]
    s_tokens = [f"s@{lvl}" for lvl in s_lvls[1:]]   # non-baseline
    m_tokens = [f"m@{lvl}" for lvl in m_lvls[1:]]
    g_tokens = [f"g@{lvl}" for lvl in g_lvls[1:]]
    return dict(
        s_base=s_base, m_base=m_base, g_base=g_base,
        s_levels=s_lvls, m_levels=m_lvls, g_levels=g_lvls,
        s_tokens=s_tokens, m_tokens=m_tokens, g_tokens=g_tokens
    )

def make_main_dummies(df: pd.DataFrame, toks):
    D = pd.DataFrame(index=df.index)
    for t in toks["s_tokens"]:
        lvl = t.split("@",1)[1]; D[t] = (df["session"] == lvl).astype(int)
    for t in toks["m_tokens"]:
        lvl = t.split("@",1)[1]; D[t] = (df["modality"] == lvl).astype(int)
    for t in toks["g_tokens"]:
        lvl = t.split("@",1)[1]; D[t] = (df["group"] == lvl).astype(int)
    return D

def add_interactions(D: pd.DataFrame, toks, orders=(("s","m"),("s","g"),("m","g")), three_way=True):
    out = D.copy()
    pref = {"s": toks["s_tokens"], "m": toks["m_tokens"], "g": toks["g_tokens"]}
    # 2-way
    for a, b in orders:
        for ta, tb in product(pref[a], pref[b]):
            out[f"{ta}*{tb}"] = D[ta] * D[tb]
    # 3-way
    if three_way and toks["s_tokens"] and toks["m_tokens"] and toks["g_tokens"]:
        for ts, tm, tg in product(toks["s_tokens"], toks["m_tokens"], toks["g_tokens"]):
            out[f"{ts}*{tm}*{tg}"] = D[ts] * D[tm] * D[tg]
    return out

def schema_for_level_cp(df, level:int):
    toks  = factor_tokens(df)
    mains = make_main_dummies(df, toks)

    S, M, G = toks["s_tokens"], toks["m_tokens"], toks["g_tokens"]
    ALL_2W = add_interactions(mains, toks, three_way=False).columns.difference(mains.columns).tolist()
    SM = [c for c in ALL_2W if c.startswith("s@") and "*m@" in c]
    SG = [c for c in ALL_2W if c.startswith("s@") and "*g@" in c]
    MG = [c for c in ALL_2W if c.startswith("m@") and "*g@" in c]
    ALL_3W = add_interactions(mains, toks, three_way=True).columns.difference(mains.columns).difference(ALL_2W).tolist()

    if level == 0:
        cols = []
    elif level == 1:
        cols = S
    elif level == 2:
        cols = S + M
    elif level == 3:
        cols = S + G
    elif level == 4:
        cols = S + M + SM
    elif level == 5:
        cols = S + G + SG
    elif level == 6:
        cols = S + M + G
    elif level == 7:
        cols = S + M + G + SM + SG + MG
    elif level == 8:
        cols = S + M + G + SM + SG + MG + ALL_3W
    else:
        raise ValueError("level must be 0..8")

    if cols:
        D_full = add_interactions(mains, toks, three_way=(level==8))
        X = D_full[cols].to_numpy(dtype=float)
    else:
        X = np.zeros((len(df), 0), dtype=float)

    return dict(X=X, beta_names=cols, tokens=toks)

def build_design_for_level_cp(df: pd.DataFrame, level:int):
    dm = schema_for_level_cp(df, level)
    return dict(
        N=len(df),
        K=dm["X"].shape[1],
        y=df["y"].astype(float).to_numpy(),
        X=dm["X"],
        beta_names=dm["beta_names"],
        tokens=dm["tokens"],
    )

# ---------- Stan: CP heteroscedastic (separate μ and σ designs) ----------
CP_HET_STAN = r"""
data {
  int<lower=1> N;
  int<lower=0> Kmu;
  int<lower=0> Ksig;
  vector[N] y;
  matrix[N, Kmu] Xmu;     // may be N×0
  matrix[N, Ksig] Xsig;   // may be N×0
}
parameters {
  real alpha;                       // intercept for mean
  vector[Kmu] beta_mu;              // coefficients for mean
  real<lower=0> sigma0;             // baseline SD
  vector[Ksig] gamma_sig;           // coefficients on log-variance
}
transformed parameters {
  vector[N] mu;
  vector[N] sigma;
  // mean
  if (Kmu == 0) {
    mu = rep_vector(alpha, N);
  } else {
    mu = alpha + Xmu * beta_mu;
  }
  // variance via log link: sigma = sigma0 * exp(0.5 * (Xsig * gamma_sig))
  if (Ksig == 0) {
    sigma = rep_vector(sigma0, N);
  } else {
    vector[N] eta = Xsig * gamma_sig;
    sigma = sigma0 * exp(0.5 * eta);
  }
}
model {
  // weakly-informative priors
  alpha ~ normal(0, 5);
  if (Kmu > 0) beta_mu ~ normal(0, 2);
  sigma0 ~ normal(0, 1);            // half-normal via <lower=0>
  if (Ksig > 0) gamma_sig ~ normal(0, 1);

  y ~ normal(mu, sigma);
}
generated quantities {
  vector[N] log_lik;
  for (n in 1:N) {
    log_lik[n] = normal_lpdf(y[n] | mu[n], sigma[n]);
  }
}
""".strip()

(STAN_DIR / "cp_hetero_kmu_ksig.stan").write_text(CP_HET_STAN)
m_cp_het = cs.CmdStanModel(stan_file=str(STAN_DIR/"cp_hetero_kmu_ksig.stan"))

# ---------- Fit helper ----------
def _empty_n_by_0(n: int):
    return np.empty((n, 0), dtype=float).tolist()

def fit_cp_hetero(level:int, df: pd.DataFrame, use_same_for_sigma: bool=True, cols_for_sigma=None):
    dm_mu = build_design_for_level_cp(df, level)
    N, Kmu, y, Xmu = dm_mu["N"], dm_mu["K"], dm_mu["y"], dm_mu["X"]

    if use_same_for_sigma:
        Ksig = Kmu
        Xsig = Xmu
        beta_names_sig = dm_mu["beta_names"]
    else:
        toks  = dm_mu["tokens"]
        mains = make_main_dummies(df, toks)
        Dfull = add_interactions(mains, toks, three_way=(level==8))
        if cols_for_sigma is None:
            cols_for_sigma = []
        if cols_for_sigma:
            Xsig = Dfull[cols_for_sigma].to_numpy(dtype=float)
        else:
            Xsig = np.empty((N,0), dtype=float)
        Ksig = Xsig.shape[1]
        beta_names_sig = cols_for_sigma

    data = dict(
        N=int(N),
        Kmu=int(Kmu),
        Ksig=int(Ksig),
        y=y.tolist(),
        Xmu=(Xmu.tolist() if Kmu > 0 else _empty_n_by_0(N)),
        Xsig=(Xsig.tolist() if Ksig > 0 else _empty_n_by_0(N)),
    )

    res_dir = RES_DIR / f"CP{level}"
    if res_dir.exists():
        shutil.rmtree(res_dir)
    res_dir.mkdir(parents=True, exist_ok=True)

    t0 = time.perf_counter()
    fit = m_cp_het.sample(
        data=data,
        seed=SEED, chains=CHAINS, parallel_chains=PARALLEL,
        iter_warmup=WARMUP, iter_sampling=DRAWS, thin=THIN,
        show_progress=True
    )
    dt = time.perf_counter() - t0

    idata = az.from_cmdstanpy(posterior=fit, log_likelihood="log_lik")

    # annotate names for later visualization
    if Kmu > 0:
        idata.posterior = idata.posterior.assign_coords({"beta_mu_name": dm_mu["beta_names"]})
    if Ksig > 0:
        idata.posterior = idata.posterior.assign_coords({"beta_sig_name": beta_names_sig})

    out_nc = res_dir / "posterior.nc"
    az.to_netcdf(idata, out_nc)

    meta = dict(
        level=level, seed=SEED,
        chains=CHAINS, warmup=WARMUP, draws=DRAWS, thin=THIN,
        N=int(N), Kmu=int(Kmu), Ksig=int(Ksig),
        beta_mu_names=(dm_mu["beta_names"] if Kmu>0 else []),
        beta_sig_names=(beta_names_sig if Ksig>0 else []),
        baselines=dict(
            session=dm_mu["tokens"]["s_base"],
            modality=dm_mu["tokens"]["m_base"],
            group=dm_mu["tokens"]["g_base"],
        ),
        labels=dict(
            session=dm_mu["tokens"]["s_levels"],
            modality=dm_mu["tokens"]["m_levels"],
            group=dm_mu["tokens"]["g_levels"],
        )
    )
    (res_dir/"meta.json").write_text(json.dumps(meta, indent=2))
    return idata, meta

In [ ]:
# Load data and restrict to Pre/Post1
df = load_long(DATA_PATH)
sess_levels = list(df["session"].cat.categories)
PRE, POST1 = sess_levels[0], sess_levels[1]

df_fit = df[df["session"].isin([PRE, POST1])].copy()
df_fit["session"] = df_fit["session"].cat.remove_unused_categories()

# Fit the full CP family requested for this session
SELECTED_MODELS = list(range(1, 9))

fitted_cp = {}
timings_cp = {}

for L in SELECTED_MODELS:
    t0 = time.perf_counter()
    idata, meta = fit_cp_hetero(level=L, df=df_fit, use_same_for_sigma=True)
    t1 = time.perf_counter()
    fitted_cp[L] = (idata, meta)
    timings_cp[L] = t1 - t0

pd.DataFrame.from_dict(timings_cp, orient="index", columns=["seconds"])\
  .rename_axis("CP_model")\
  .to_csv(RES_DIR / "CP_model_fit_times.csv")

In [ ]:
# Model comparison for the fitted CP family
compare_inputs = {f"CP{L}": fitted_cp[L][0] for L in SELECTED_MODELS}
cp_compare = az.compare(compare_inputs, ic="loo", method="stacking")
cp_compare["looic"] = -2.0 * cp_compare["elpd_loo"]
cp_compare = cp_compare[["rank", "elpd_loo", "looic", "p_loo", "dse", "weight"]]
cp_compare.to_csv(RES_DIR / "CP_model_comparison_loo.csv")
print(cp_compare)
best_cp = cp_compare.sort_values("rank").index[0]
print(f"Best CP model by PSIS-LOO: {best_cp}")
